In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import pickle
import numpy as np
from tqdm.notebook import tqdm
from tensorflow.keras.applications.vgg16 import VGG16,preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical,plot_model
from tensorflow.keras.layers import Input,Dense,LSTM,Embedding,Dropout,add

In [ ]:
base_dir = "/kaggle/input/flickr8k"
working_dir = "/kaggle/working/"

# **VGG MODEL**

In [ ]:
model = VGG16()

model = Model(inputs=model.inputs,outputs=model.layers[-2].output)
model.summary()

# **image feature extraction**

In [ ]:
features = {}

img_dir = os.path.join(base_dir,'Images')

for img_name in tqdm(os.listdir(img_dir)):

    img_path = img_dir + "/" + img_name

    img = load_img(img_path,target_size=(224,224))

    img = img_to_array(img)

    img = img.reshape(1,img.shape[0],img.shape[1],img.shape[2])

    img = preprocess_input(img)

    feature = model.predict(img)

    img_id = img_name.split('.')[0]

    features[img_id] = feature
    

In [ ]:
pickle.dump(features,open(os.path.join(working_dir,"features.pkl"),'wb'))

with open(os.path.join(working_dir,"features.pkl"),'rb') as file:
    features = pickle.load(file)

In [ ]:
with open(os.path.join(base_dir,'captions.txt'),'r') as File:

    next(File)

    captions_document = File.read()

In [ ]:
mappings = {}

for line in tqdm(captions_document.split("\n")):

    tokens = line.split(",")
    # print(tokens)

    if len(line) > 2:
        continue

    image_id,caption = tokens[0],tokens[1:]

    image_id = image_id.split(".")[0]

    caption = "".join(caption)

    if image_id not in mappings:

        mappings[image_id] = []

    mappings[image_id].append(caption)

In [ ]:
def clean_mapping(mappings):

    for img_id,captions in mappings.items():

        for i in range(len(captions)):
            
            caption = captions[i].lower()
            caption = caption.replace('[^A-Za-z]', '')
            caption = caption.replace('\s+', ' ')
            caption = "startseq " + " ".join([word for word in caption.split() if len(word) > 1]) + " endseq"
            captions[i] = caption

clean_mapping(mappings)

In [ ]:
All_captions = [caption for key in mappings for caption in mappings[key]]

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(All_captions)
vacab = len(tokenizer.word_index) + 1
max_length = max(len(caption.split())for caption in All_captions)

In [ ]:
image_ids = list(mappings.keys())
split = int(len(image_ids) * 0.9)
train = image_ids[:split]
test = image_ids[split:]

In [ ]:
def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = list(), list(), list()
    n = 0
    while 1:
        for key in data_keys:
            captions = mapping[key]
            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    X1.append(features[key][0])
                    X2.append(in_seq)
                    y.append(out_seq)
            n += 1
            if n == batch_size:
                yield {"image": np.array(X1), "text": np.array(X2)}, np.array(y)
                X1, X2, y = list(), list(), list()
                n = 0

In [ ]:
inputs1=Input(shape=(4096,),name="image")
fe1=Dropout(0.4)(inputs1)
fe2=Dense(256,activation='relu')(fe1)

inputs2=Input(shape=(max_length,),name="text")
se1=Embedding(vacab,256,mask_zero=False)(inputs2)
se2=Dropout(0.4)(se1)
se3=LSTM(256,return_sequences=False,use_cudnn=False)(se2)



decoder1=add([fe2,se3])
decoder2=Dense(256,activation='relu')(decoder1)
outputs=Dense(vacab,activation='softmax')(decoder2)

model=Model(inputs=[inputs1,inputs2],outputs=outputs)
model.compile(loss='categorical_crossentropy',optimizer='adam')

model.summary()

In [ ]:
epochs=10
batch_size=32
steps=len(train)//batch_size

for i in range(epochs):
    generator=data_generator(train,mappings,features,tokenizer,max_length,vacab,batch_size)
    model.fit(generator, epochs=1,steps_per_epoch=steps,verbose=1)

In [ ]:
model.save(working_dir + '/Imagetocaption_model.h5')

In [ ]:
def idx_to_word(integer, tokenizer):
    for word, index in tokenizer.word_index.items():
        if index == integer:
            return word
    return None


def predict_caption(model, image, tokenizer, max_length):
    in_text = 'startseq'
    for i in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], max_length)
        yhat = model.predict([image, sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = idx_to_word(yhat, tokenizer)
        if word is None:
            break
        in_text += " " + word
        if word == 'endseq':
            break
      
    return in_text


from PIL import Image
import matplotlib.pyplot as plt
def generate_caption(image_name):
    # load the image
    
    img_path = image_name
    image = Image.open(img_path)
    captions = mappings[image_id]
    print('---------------------Actual---------------------')
    for caption in captions:
        print(caption)
    # predict the caption
    y_pred = predict_caption(model, features[image_id], tokenizer, max_length)
    print('--------------------Predicted--------------------')
    print(y_pred)
    plt.imshow(image)

In [ ]:
generate_caption("/kaggle/input/flickr8k/Images/1000268201_693b08cb0e.jpg")